In [ ]:

# ================================================================
# SupplyChainAI — 01 DATA EXPLORATION
# Single-Cell Jupyter Notebook
# ================================================================
# Features:
#  1. Load retail supply chain CSV datasets
#  2. Generate synthetic demo data if files are missing
#  3. Dataset profiling and data quality analysis
#  4. Sales and demand exploration
#  5. Inventory and stockout analysis
#  6. Supplier performance and lead-time analysis
#  7. Product/category analytics
#  8. Correlation and anomaly detection
#  9. Visual dashboards
# 10. Export EDA reports and cleaned datasets
#
# Required:
#   pip install pandas numpy matplotlib seaborn scikit-learn
# ================================================================

from pathlib import Path
from datetime import datetime, timedelta
import warnings
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import IsolationForest

warnings.filterwarnings("ignore", category=FutureWarning)

# ------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------

PROJECT_NAME = "SupplyChainAI"
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_DIR = Path("data")
REPORT_DIR = Path("reports/eda")
OUTPUT_DIR = Path("data/processed")

for directory in [DATA_DIR, REPORT_DIR, OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

SALES_FILE = DATA_DIR / "sales.csv"
INVENTORY_FILE = DATA_DIR / "inventory.csv"
SUPPLIERS_FILE = DATA_DIR / "suppliers.csv"
PRODUCTS_FILE = DATA_DIR / "products.csv"

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 110

print("=" * 70)
print("SUPPLYCHAINAI — DATA EXPLORATION")
print("=" * 70)

# ------------------------------------------------
# 2. SYNTHETIC DEMO DATA GENERATION
# ------------------------------------------------

def generate_demo_data():
    """Generate fictional retail supply-chain datasets."""
    rng = np.random.default_rng(RANDOM_SEED)

    n_products = 120
    n_suppliers = 20
    n_warehouses = 5
    n_days = 180

    categories = [
        "Electronics", "Groceries", "Fashion",
        "Home", "Beauty", "Sports"
    ]

    product_ids = [f"SKU-{i:04d}" for i in range(1, n_products + 1)]
    supplier_ids = [f"SUP-{i:03d}" for i in range(1, n_suppliers + 1)]
    warehouse_ids = [f"WH-{i:02d}" for i in range(1, n_warehouses + 1)]

    products = pd.DataFrame({
        "product_id": product_ids,
        "product_name": [f"Product {i}" for i in range(1, n_products + 1)],
        "category": rng.choice(categories, n_products),
        "supplier_id": rng.choice(supplier_ids, n_products),
        "unit_price": np.round(rng.uniform(100, 6000, n_products), 2),
        "unit_cost": np.round(rng.uniform(50, 90, n_products), 2)
    })

    # Ensure cost is below selling price.
    products["unit_cost"] = np.round(
        products["unit_price"] * rng.uniform(0.45, 0.80, n_products), 2
    )

    suppliers = pd.DataFrame({
        "supplier_id": supplier_ids,
        "supplier_name": [f"Supplier {i}" for i in range(1, n_suppliers + 1)],
        "lead_time_days": rng.integers(2, 22, n_suppliers),
        "on_time_delivery_rate": np.round(
            rng.uniform(0.70, 0.99, n_suppliers), 3
        ),
        "defect_rate": np.round(rng.uniform(0.001, 0.09, n_suppliers), 3),
        "risk_score": rng.integers(5, 90, n_suppliers)
    })

    inventory = pd.DataFrame({
        "product_id": np.repeat(product_ids, n_warehouses),
        "warehouse_id": np.tile(warehouse_ids, n_products),
        "stock_on_hand": rng.integers(0, 800, n_products * n_warehouses),
        "reorder_point": rng.integers(40, 180, n_products * n_warehouses),
        "safety_stock": rng.integers(15, 100, n_products * n_warehouses)
    })

    dates = pd.date_range(
        end=pd.Timestamp.today().normalize(),
        periods=n_days
    )

    sales_rows = []

    base_demand = rng.integers(2, 30, n_products)

    for day in dates:
        weekend_multiplier = 1.25 if day.dayofweek >= 5 else 1.0

        for idx, product_id in enumerate(product_ids):
            # Randomly omit some product/day observations.
            if rng.random() < 0.12:
                continue

            seasonal_factor = (
                1.0 + 0.18 * np.sin(2 * np.pi * day.dayofyear / 365)
            )

            demand = rng.poisson(
                max(
                    0.1,
                    base_demand[idx]
                    * weekend_multiplier
                    * seasonal_factor
                )
            )

            sales_rows.append({
                "date": day,
                "product_id": product_id,
                "warehouse_id": rng.choice(warehouse_ids),
                "quantity_sold": int(demand)
            })

    sales = pd.DataFrame(sales_rows)

    sales = sales.merge(
        products[["product_id", "unit_price"]],
        on="product_id",
        how="left"
    )

    sales["revenue"] = (
        sales["quantity_sold"] * sales["unit_price"]
    ).round(2)

    products.to_csv(PRODUCTS_FILE, index=False)
    suppliers.to_csv(SUPPLIERS_FILE, index=False)
    inventory.to_csv(INVENTORY_FILE, index=False)
    sales.to_csv(SALES_FILE, index=False)

    print("Synthetic demonstration datasets generated.")
    print("These records are fictional, not real business data.")

    return sales, inventory, suppliers, products


# ------------------------------------------------
# 3. LOAD DATA
# ------------------------------------------------

required_files = [
    SALES_FILE,
    INVENTORY_FILE,
    SUPPLIERS_FILE,
    PRODUCTS_FILE
]

if all(path.exists() for path in required_files):
    print("\nLoading existing CSV datasets...")

    sales = pd.read_csv(SALES_FILE)
    inventory = pd.read_csv(INVENTORY_FILE)
    suppliers = pd.read_csv(SUPPLIERS_FILE)
    products = pd.read_csv(PRODUCTS_FILE)

    DATA_SOURCE = "CSV files"
else:
    # Do not overwrite a partially populated real dataset folder.
    if any(path.exists() for path in required_files):
        missing = [str(p) for p in required_files if not p.exists()]
        raise FileNotFoundError(
            "Some required CSV files are missing: "
            + ", ".join(missing)
            + ". Supply all four files or use an empty data folder "
              "to generate fictional demonstration data."
        )

    sales, inventory, suppliers, products = generate_demo_data()
    DATA_SOURCE = "Synthetic demonstration data"

print(f"\nData source: {DATA_SOURCE}")

datasets = {
    "Sales": sales,
    "Inventory": inventory,
    "Suppliers": suppliers,
    "Products": products
}

# ------------------------------------------------
# 4. DATASET SUMMARY
# ------------------------------------------------

print("\nDATASET SUMMARY")
print("-" * 70)

summary_rows = []

for name, df in datasets.items():
    summary_rows.append({
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_values": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
        "memory_mb": round(
            df.memory_usage(deep=True).sum() / (1024 ** 2), 3
        )
    })

dataset_summary = pd.DataFrame(summary_rows)
display(dataset_summary)

# ------------------------------------------------
# 5. SCHEMA VALIDATION
# ------------------------------------------------

required_columns = {
    "Sales": ["date", "product_id", "quantity_sold"],
    "Inventory": [
        "product_id", "warehouse_id",
        "stock_on_hand", "reorder_point", "safety_stock"
    ],
    "Suppliers": [
        "supplier_id", "lead_time_days",
        "on_time_delivery_rate", "defect_rate"
    ],
    "Products": [
        "product_id", "category", "supplier_id", "unit_price"
    ]
}

for name, required in required_columns.items():
    missing = [
        col for col in required
        if col not in datasets[name].columns
    ]
    if missing:
        raise ValueError(
            f"{name} dataset missing required columns: {missing}"
        )

sales["date"] = pd.to_datetime(sales["date"], errors="coerce")

numeric_fields = {
    "Sales": ["quantity_sold"],
    "Inventory": [
        "stock_on_hand", "reorder_point", "safety_stock"
    ],
    "Suppliers": [
        "lead_time_days", "on_time_delivery_rate", "defect_rate"
    ],
    "Products": ["unit_price"]
}

for name, fields in numeric_fields.items():
    for col in fields:
        datasets[name][col] = pd.to_numeric(
            datasets[name][col], errors="coerce"
        )

# Optional columns
if "unit_cost" in products.columns:
    products["unit_cost"] = pd.to_numeric(
        products["unit_cost"], errors="coerce"
    )

if "risk_score" in suppliers.columns:
    suppliers["risk_score"] = pd.to_numeric(
        suppliers["risk_score"], errors="coerce"
    )

# ------------------------------------------------
# 6. DATA QUALITY
# ------------------------------------------------

quality_rows = []

for name, df in datasets.items():
    for col in df.columns:
        quality_rows.append({
            "dataset": name,
            "column": col,
            "dtype": str(df[col].dtype),
            "missing": int(df[col].isna().sum()),
            "missing_pct": round(
                100 * df[col].isna().mean(), 2
            ) if len(df) else 0,
            "unique_values": int(df[col].nunique(dropna=True))
        })

quality_report = pd.DataFrame(quality_rows)

print("\nDATA QUALITY REPORT")
display(quality_report)

# Data integrity checks
integrity = {
    "sales_invalid_dates": int(sales["date"].isna().sum()),
    "sales_negative_quantity": int(
        (sales["quantity_sold"] < 0).sum()
    ),
    "inventory_negative_stock": int(
        (inventory["stock_on_hand"] < 0).sum()
    ),
    "products_nonpositive_price": int(
        (products["unit_price"] <= 0).sum()
    ),
    "suppliers_invalid_delivery_rate": int(
        (
            (suppliers["on_time_delivery_rate"] < 0) |
            (suppliers["on_time_delivery_rate"] > 1)
        ).sum()
    ),
    "suppliers_invalid_defect_rate": int(
        (
            (suppliers["defect_rate"] < 0) |
            (suppliers["defect_rate"] > 1)
        ).sum()
    ),
    "unknown_sales_products": int(
        (~sales["product_id"].isin(products["product_id"])).sum()
    )
}

print("\nDATA INTEGRITY CHECKS")
display(pd.DataFrame(
    integrity.items(),
    columns=["check", "invalid_records"]
))

# ------------------------------------------------
# 7. CLEAN ANALYSIS COPIES
# ------------------------------------------------

sales_clean = sales.copy()
inventory_clean = inventory.copy()
suppliers_clean = suppliers.copy()
products_clean = products.copy()

sales_clean = sales_clean.dropna(
    subset=["date", "product_id", "quantity_sold"]
)
sales_clean = sales_clean[
    sales_clean["quantity_sold"] >= 0
].copy()

inventory_clean = inventory_clean.dropna(
    subset=[
        "product_id", "warehouse_id",
        "stock_on_hand", "reorder_point"
    ]
)
inventory_clean = inventory_clean[
    (inventory_clean["stock_on_hand"] >= 0) &
    (inventory_clean["reorder_point"] >= 0)
].copy()

products_clean = products_clean.dropna(
    subset=["product_id", "unit_price"]
)
products_clean = products_clean[
    products_clean["unit_price"] > 0
].copy()

suppliers_clean = suppliers_clean.dropna(
    subset=["supplier_id", "lead_time_days"]
)
suppliers_clean = suppliers_clean[
    suppliers_clean["lead_time_days"] >= 0
].copy()

# Avoid ambiguous many-to-many joins.
if products_clean["product_id"].duplicated().any():
    raise ValueError("Products contains duplicate product_id values.")

if suppliers_clean["supplier_id"].duplicated().any():
    raise ValueError("Suppliers contains duplicate supplier_id values.")

sales_clean = sales_clean.merge(
    products_clean[
        ["product_id", "category", "unit_price", "supplier_id"]
    ],
    on="product_id",
    how="left",
    validate="many_to_one",
    suffixes=("", "_product")
)

# Recalculate revenue using product master prices.
sales_clean["revenue"] = (
    sales_clean["quantity_sold"] * sales_clean["unit_price"]
).round(2)

# ------------------------------------------------
# 8. BUSINESS KPIs
# ------------------------------------------------

total_revenue = float(sales_clean["revenue"].sum())
total_units = float(sales_clean["quantity_sold"].sum())
total_products = int(products_clean["product_id"].nunique())
total_suppliers = int(suppliers_clean["supplier_id"].nunique())
total_inventory = float(inventory_clean["stock_on_hand"].sum())

inventory_clean["below_reorder"] = (
    inventory_clean["stock_on_hand"] <
    inventory_clean["reorder_point"]
)

low_stock_count = int(inventory_clean["below_reorder"].sum())

kpis = pd.DataFrame([
    {"metric": "Total Revenue", "value": round(total_revenue, 2)},
    {"metric": "Units Sold", "value": int(total_units)},
    {"metric": "Products", "value": total_products},
    {"metric": "Suppliers", "value": total_suppliers},
    {"metric": "Inventory Units", "value": int(total_inventory)},
    {"metric": "Low-Stock Locations", "value": low_stock_count}
])

print("\nSUPPLY CHAIN KPIs")
display(kpis)

# ------------------------------------------------
# 9. DAILY SALES & DEMAND
# ------------------------------------------------

daily_sales = (
    sales_clean.groupby("date", as_index=False)
    .agg(
        units_sold=("quantity_sold", "sum"),
        revenue=("revenue", "sum")
    )
    .sort_values("date")
)

daily_sales["rolling_demand_7d"] = (
    daily_sales["units_sold"]
    .rolling(window=7, min_periods=1)
    .mean()
)

daily_sales["rolling_demand_30d"] = (
    daily_sales["units_sold"]
    .rolling(window=30, min_periods=1)
    .mean()
)

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(
    daily_sales["date"],
    daily_sales["units_sold"],
    alpha=0.45,
    label="Daily Units Sold"
)

ax.plot(
    daily_sales["date"],
    daily_sales["rolling_demand_7d"],
    linewidth=2,
    label="7-Day Moving Average"
)

ax.plot(
    daily_sales["date"],
    daily_sales["rolling_demand_30d"],
    linewidth=2,
    label="30-Day Moving Average"
)

ax.set_title("Daily Sales Volume & Rolling Trends")
ax.set_xlabel("Date")
ax.set_ylabel("Units")
ax.legend()
plt.tight_layout()
plt.savefig(REPORT_DIR / "daily_demand.png")
plt.show()

# ------------------------------------------------
# 10. CATEGORY PERFORMANCE
# ------------------------------------------------

category_sales = (
    sales_clean.groupby("category", dropna=False, as_index=False)
    .agg(
        units_sold=("quantity_sold", "sum"),
        revenue=("revenue", "sum")
    )
    .sort_values("revenue", ascending=False)
)

print("\nCATEGORY PERFORMANCE")
display(category_sales)

plt.figure(figsize=(12, 5))
sns.barplot(
    data=category_sales,
    x="category",
    y="revenue"
)
plt.title("Revenue by Product Category")
plt.xticks(rotation=25)
plt.tight_layout()
plt.savefig(REPORT_DIR / "category_revenue.png")
plt.show()

# ------------------------------------------------
# 11. TOP SELLING PRODUCTS
# ------------------------------------------------

top_products = (
    sales_clean.groupby("product_id", as_index=False)
    .agg(
        units_sold=("quantity_sold", "sum"),
        revenue=("revenue", "sum")
    )
    .sort_values("revenue", ascending=False)
    .head(15)
)

print("\nTOP 15 PRODUCTS")
display(top_products)

plt.figure(figsize=(12, 6))
sns.barplot(
    data=top_products,
    y="product_id",
    x="revenue"
)
plt.title("Top 15 Products by Revenue")
plt.tight_layout()
plt.savefig(REPORT_DIR / "top_products.png")
plt.show()

# ------------------------------------------------
# 12. INVENTORY INTELLIGENCE
# ------------------------------------------------

inventory_analysis = inventory_clean.merge(
    products_clean[
        ["product_id", "category", "unit_price"]
    ],
    on="product_id",
    how="left",
    validate="many_to_one"
)

inventory_analysis["stock_gap"] = (
    inventory_analysis["stock_on_hand"] -
    inventory_analysis["reorder_point"]
)

inventory_analysis["stock_status"] = np.select(
    [
        inventory_analysis["stock_on_hand"] <= 0,
        inventory_analysis["stock_on_hand"] <
            inventory_analysis["reorder_point"]
    ],
    [
        "Stockout",
        "Low Stock"
    ],
    default="Healthy"
)

inventory_analysis["inventory_retail_value"] = (
    inventory_analysis["stock_on_hand"] *
    inventory_analysis["unit_price"]
)

stock_status = (
    inventory_analysis["stock_status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="count")
)

print("\nINVENTORY STATUS")
display(stock_status)

plt.figure(figsize=(8, 5))
sns.barplot(
    data=stock_status,
    x="status",
    y="count"
)
plt.title("Inventory Status Distribution")
plt.tight_layout()
plt.savefig(REPORT_DIR / "inventory_status.png")
plt.show()

# ------------------------------------------------
# 13. STOCK COVERAGE
# ------------------------------------------------

product_demand = (
    sales_clean.groupby("product_id", as_index=False)
    .agg(total_units=("quantity_sold", "sum"))
)

observation_days = max(
    1,
    (
        sales_clean["date"].max() -
        sales_clean["date"].min()
    ).days + 1
) if len(sales_clean) else 1

product_demand["avg_daily_sales"] = (
    product_demand["total_units"] / observation_days
)

product_inventory = (
    inventory_analysis.groupby("product_id", as_index=False)
    .agg(stock_on_hand=("stock_on_hand", "sum"))
)

coverage = product_inventory.merge(
    product_demand,
    on="product_id",
    how="left"
)

coverage["avg_daily_sales"] = (
    coverage["avg_daily_sales"].fillna(0)
)

coverage["days_of_cover"] = np.where(
    coverage["avg_daily_sales"] > 0,
    coverage["stock_on_hand"] /
        coverage["avg_daily_sales"].replace(0, np.nan),
    np.nan
)

coverage["coverage_status"] = np.select(
    [
        coverage["stock_on_hand"] <= 0,
        coverage["days_of_cover"] < 7,
        coverage["days_of_cover"] < 14
    ],
    [
        "Stockout",
        "Critical",
        "Low"
    ],
    default="Healthy / Unknown"
)

print("\nLOWEST STOCK COVERAGE")
display(
    coverage.sort_values(
        "days_of_cover",
        na_position="last"
    ).head(15)
)

# ------------------------------------------------
# 14. SUPPLIER PERFORMANCE
# ------------------------------------------------

supplier_analysis = suppliers_clean.copy()

supplier_analysis["delivery_performance_pct"] = (
    supplier_analysis["on_time_delivery_rate"] * 100
)

supplier_analysis["defect_rate_pct"] = (
    supplier_analysis["defect_rate"] * 100
)

if "risk_score" not in supplier_analysis.columns:
    supplier_analysis["risk_score"] = (
        (1 - supplier_analysis["on_time_delivery_rate"]) * 50
        + supplier_analysis["defect_rate"] * 250
        + supplier_analysis["lead_time_days"] * 1.2
    ).clip(0, 100)

supplier_analysis["risk_category"] = pd.cut(
    supplier_analysis["risk_score"],
    bins=[-np.inf, 25, 45, 65, np.inf],
    labels=["Low", "Medium", "High", "Critical"],
    right=False
)

print("\nSUPPLIER RISK ANALYSIS")
display(
    supplier_analysis.sort_values(
        "risk_score",
        ascending=False
    ).head(15)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(
    supplier_analysis["lead_time_days"],
    bins=12,
    kde=True,
    ax=axes[0]
)
axes[0].set_title("Supplier Lead Time Distribution")

sns.scatterplot(
    data=supplier_analysis,
    x="lead_time_days",
    y="delivery_performance_pct",
    hue="risk_category",
    s=90,
    ax=axes[1]
)
axes[1].set_title("Lead Time vs On-Time Delivery")

plt.tight_layout()
plt.savefig(REPORT_DIR / "supplier_performance.png")
plt.show()

# ------------------------------------------------
# 15. DAY-OF-WEEK DEMAND
# ------------------------------------------------

sales_clean["day_of_week"] = sales_clean["date"].dt.day_name()

weekday_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

weekday_sales = (
    sales_clean.groupby("day_of_week", as_index=False)
    .agg(units_sold=("quantity_sold", "sum"))
)

plt.figure(figsize=(12, 5))
sns.barplot(
    data=weekday_sales,
    x="day_of_week",
    y="units_sold",
    order=weekday_order
)
plt.title("Sales Volume by Day of Week")
plt.xticks(rotation=25)
plt.tight_layout()
plt.savefig(REPORT_DIR / "weekday_demand.png")
plt.show()

# ------------------------------------------------
# 16. CORRELATION ANALYSIS
# ------------------------------------------------

numeric_inventory = inventory_analysis.select_dtypes(
    include=np.number
)

correlations = numeric_inventory.corr()

if not correlations.empty:
    plt.figure(figsize=(10, 7))
    sns.heatmap(
        correlations,
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        center=0
    )
    plt.title("Inventory Feature Correlations")
    plt.tight_layout()
    plt.savefig(REPORT_DIR / "correlations.png")
    plt.show()

# ------------------------------------------------
# 17. DEMAND ANOMALY DETECTION
# ------------------------------------------------

anomaly_data = daily_sales.copy()

if len(anomaly_data) >= 20:
    anomaly_features = anomaly_data[
        ["units_sold", "revenue"]
    ].fillna(0)

    detector = IsolationForest(
        n_estimators=150,
        contamination=0.05,
        random_state=RANDOM_SEED
    )

    anomaly_data["anomaly_flag"] = detector.fit_predict(
        anomaly_features
    )

    anomaly_data["is_anomaly"] = (
        anomaly_data["anomaly_flag"] == -1
    )

    plt.figure(figsize=(14, 5))

    plt.plot(
        anomaly_data["date"],
        anomaly_data["units_sold"],
        label="Daily Sales"
    )

    anomalies = anomaly_data[
        anomaly_data["is_anomaly"]
    ]

    plt.scatter(
        anomalies["date"],
        anomalies["units_sold"],
        color="red",
        label="Statistical Anomalies",
        zorder=5
    )

    plt.title("Daily Sales Anomaly Detection")
    plt.xlabel("Date")
    plt.ylabel("Units Sold")
    plt.legend()
    plt.tight_layout()
    plt.savefig(REPORT_DIR / "demand_anomalies.png")
    plt.show()

    print("\nDETECTED STATISTICAL ANOMALIES")
    display(
        anomalies[
            ["date", "units_sold", "revenue"]
        ]
    )
else:
    anomaly_data["is_anomaly"] = False
    print("Not enough daily observations for anomaly detection.")

# ------------------------------------------------
# 18. ABC PRODUCT ANALYSIS
# ------------------------------------------------

abc_analysis = (
    sales_clean.groupby("product_id", as_index=False)
    .agg(revenue=("revenue", "sum"))
    .sort_values("revenue", ascending=False)
)

total_abc_revenue = abc_analysis["revenue"].sum()

if total_abc_revenue > 0:
    abc_analysis["cumulative_revenue_pct"] = (
        abc_analysis["revenue"].cumsum()
        / total_abc_revenue
        * 100
    )

    abc_analysis["abc_class"] = np.select(
        [
            abc_analysis["cumulative_revenue_pct"] <= 80,
            abc_analysis["cumulative_revenue_pct"] <= 95
        ],
        ["A", "B"],
        default="C"
    )
else:
    abc_analysis["cumulative_revenue_pct"] = 0
    abc_analysis["abc_class"] = "Unclassified"

print("\nABC PRODUCT ANALYSIS")
display(abc_analysis.head(20))

# ------------------------------------------------
# 19. EXPORT CLEANED DATA
# ------------------------------------------------

sales_clean.to_csv(
    OUTPUT_DIR / "sales_clean.csv",
    index=False
)

inventory_analysis.to_csv(
    OUTPUT_DIR / "inventory_analysis.csv",
    index=False
)

supplier_analysis.to_csv(
    OUTPUT_DIR / "supplier_analysis.csv",
    index=False
)

coverage.to_csv(
    OUTPUT_DIR / "stock_coverage.csv",
    index=False
)

abc_analysis.to_csv(
    OUTPUT_DIR / "abc_analysis.csv",
    index=False
)

daily_sales.to_csv(
    OUTPUT_DIR / "daily_sales.csv",
    index=False
)

dataset_summary.to_csv(
    REPORT_DIR / "dataset_summary.csv",
    index=False
)

quality_report.to_csv(
    REPORT_DIR / "data_quality.csv",
    index=False
)

# ------------------------------------------------
# 20. JSON REPORT
# ------------------------------------------------

report = {
    "project": PROJECT_NAME,
    "analysis": "Retail Supply Chain Exploratory Data Analysis",
    "data_source": DATA_SOURCE,
    "generated_at": datetime.now().isoformat(),
    "datasets": {
        name.lower(): {
            "rows": int(len(df)),
            "columns": int(len(df.columns))
        }
        for name, df in datasets.items()
    },
    "kpis": {
        "total_revenue": round(total_revenue, 2),
        "units_sold": int(total_units),
        "products": total_products,
        "suppliers": total_suppliers,
        "inventory_units": int(total_inventory),
        "low_stock_locations": low_stock_count,
        "demand_anomalies": int(
            anomaly_data["is_anomaly"].sum()
        )
    },
    "data_integrity": integrity
}

with open(
    REPORT_DIR / "eda_report.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(report, file, indent=2)

# ------------------------------------------------
# 21. FINAL SUMMARY
# ------------------------------------------------

print("\n" + "=" * 70)
print("SUPPLYCHAINAI — EDA COMPLETED")
print("=" * 70)

print(f"Data source: {DATA_SOURCE}")
print(f"Total revenue: INR {total_revenue:,.2f}")
print(f"Units sold: {total_units:,.0f}")
print(f"Products: {total_products}")
print(f"Suppliers: {total_suppliers}")
print(f"Inventory units: {total_inventory:,.0f}")
print(f"Low-stock locations: {low_stock_count}")
print(f"Detected demand anomalies: {int(anomaly_data['is_anomaly'].sum())}")

print("\nGenerated outputs:")
print(f"  Cleaned datasets: {OUTPUT_DIR.resolve()}")
print(f"  EDA reports:      {REPORT_DIR.resolve()}")

print("\nSuggested next notebooks:")
print("  02_data_preprocessing.ipynb")
print("  03_demand_forecasting.ipynb")
print("  04_inventory_optimization.ipynb")
print("  05_supplier_risk_model.ipynb")

print("=" * 70)
